### import dependancies

In [2]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import client

from dotenv import load_dotenv

load_dotenv()



True

### download data from qdrant

In [3]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [ ]:

all_points = qdrant_client.scroll(collection_name="amazon-electronics-items-collection-01", limit=100, offset=None, with_vectors=False, with_payload=True) # we only have 50, 100 for sure ok

In [5]:
all_points

([Record(id=0, payload={'preprocessed_description': 'Laxihub Indoor Home Security Camera, P2 Baby Monitor with Camera and Audio, 1080P FHD, Night Vision, 2-Way Audio, AI Motion Detection Compatible with Alexa & Google Assistant ', 'image': 'https://m.media-amazon.com/images/I/51xQLCPORoL._AC_.jpg', 'rating_number': 6459, 'price': None, 'average_rating': 4.0, 'parent_asin': 'B08THC8HPD'}, vector=None, shard_key=None, order_value=None),
  Record(id=1, payload={'preprocessed_description': 'Neewer NW-760 HD Monitor, IPS 7" for Sony Canon Nikon Olympus Panasonic Cameras High Contrast: 1200:1; High Resolution: Full HD 1920x1200; High Brightness: 450cd/m2; 0.07875 x 0.07875 (mm) dot pitch offers you brilliant visual experience; 160°wide viewing angles IPS Panel; Ultra thin design 17mm thickness Marker Type(16:9,4:3,2.35:1,1.85:1); Marker Color(Red,Green,Blue,White,Black,Cyan,Purple,Yellow); Check Field (Gray/Red/Green/Blue); Image Flip (H, V, H/V); Image Freeze; Color Tepm. Focus Assist(Red,Y

In [8]:
all_context = [{"id": point.payload['parent_asin'], "text": point.payload['preprocessed_description']} for point in all_points[0]]
all_context

[{'id': 'B08THC8HPD',
  'text': 'Laxihub Indoor Home Security Camera, P2 Baby Monitor with Camera and Audio, 1080P FHD, Night Vision, 2-Way Audio, AI Motion Detection Compatible with Alexa & Google Assistant '},
 {'id': 'B01LWYDXNF',
  'text': 'Neewer NW-760 HD Monitor, IPS 7" for Sony Canon Nikon Olympus Panasonic Cameras High Contrast: 1200:1; High Resolution: Full HD 1920x1200; High Brightness: 450cd/m2; 0.07875 x 0.07875 (mm) dot pitch offers you brilliant visual experience; 160°wide viewing angles IPS Panel; Ultra thin design 17mm thickness Marker Type(16:9,4:3,2.35:1,1.85:1); Marker Color(Red,Green,Blue,White,Black,Cyan,Purple,Yellow); Check Field (Gray/Red/Green/Blue); Image Flip (H, V, H/V); Image Freeze; Color Tepm. Focus Assist(Red,Yellow,Blue,White: four color optional highlight over parts of the image in focus); False Color; Overexposure prompting; Squared segmentation; Embedded audio meter; Camera Mode Scan Mode(Over, Under); Zoom(Auto,4x,9x,16x); Anamorphic mode(1.3x,2.0x

### render a prompt to generate synthetic eval reference dataset

In [9]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [11]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, 

In [12]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[
  {
    "id": "B08THC8HPD",
    "text": "Laxihub Indoor Home Security Camera, P2 Baby Monitor with Camera and Audio, 1080P FHD, Night Vision, 2-Way Audio, AI Motion Detection Compatible with Alexa & Google Assistant "
  },
  {
    "id": "B01LWYDXNF",
    "text": "Neewer NW-760 HD Monitor, IPS 7\" for Sony Canon Nikon Olympus Panasonic Cameras High Contrast: 1200:1; High Resolution: Full HD 1920x1200; High Brightness: 450cd/m2; 0.07875 x 0.07875 (mm) dot pitch offers you brilliant visual experience; 160\u00b0wide viewing angles IPS Panel; Ultra thin design 17mm thickness Marker Type(16:9,4:3,2.35:1,1.85:1); Marker Color(Red,Green,Blue,White,Black,Cyan,Purple,Yellow); Check Field (Gray/Red/Green/Blue); Image Flip (H, V, H/V); Image Freeze; Color Tepm. Focus Assist(Red,Yellow,Blue,White: four color optional highlight over parts of the image in focus); False Color; Overexposure prompting; Squared segmentatio

In [21]:
response = openai.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    reasoning_effort="none" # we don't need to reason
)
print(response.choices[0].message.content)

[
  {
    "question": "Do you have any indoor security cameras with night vision and two-way audio?",
    "chunk_ids": ["B08THC8HPD"],
    "answer_example": "Yes, the Laxihub Indoor Home Security Camera includes 1080P FHD video, night vision, two-way audio, and AI motion detection.",
    "reasoning": "This product directly mentions it is an indoor security camera with night vision and 2-way audio, so one product is sufficient to answer."
  },
  {
    "question": "Is there a surge protector with a flat plug that would fit behind furniture?",
    "chunk_ids": ["B08DLDCJQR"],
    "answer_example": "Yes, the GE UltraPro 6 Outlet Surge Protector has a slim flat plug designed to save space behind furniture and appliances.",
    "reasoning": "The product description explicitly states it has a low-profile angled flat plug suitable for tight spaces."
  },
  {
    "question": "Do you sell a USB switch that lets me share one printer between two computers?",
    "chunk_ids": ["B09BQ5V2J5"],
    "a

In [38]:
print(response.usage)

CompletionUsage(completion_tokens=3961, prompt_tokens=11456, total_tokens=15417, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [32]:
json_output = response.choices[0].message.content
json_output = json_output.replace("```json", "").replace("```", "")

In [23]:
print(json_output)

[
  {
    "question": "Do you have any indoor security cameras with night vision and two-way audio?",
    "chunk_ids": ["B08THC8HPD"],
    "answer_example": "Yes, the Laxihub Indoor Home Security Camera includes 1080P FHD video, night vision, two-way audio, and AI motion detection.",
    "reasoning": "This product directly mentions it is an indoor security camera with night vision and 2-way audio, so one product is sufficient to answer."
  },
  {
    "question": "Is there a surge protector with a flat plug that would fit behind furniture?",
    "chunk_ids": ["B08DLDCJQR"],
    "answer_example": "Yes, the GE UltraPro 6 Outlet Surge Protector has a slim flat plug designed to save space behind furniture and appliances.",
    "reasoning": "The product description explicitly states it has a low-profile angled flat plug suitable for tight spaces."
  },
  {
    "question": "Do you sell a USB switch that lets me share one printer between two computers?",
    "chunk_ids": ["B09BQ5V2J5"],
    "a

In [33]:
json_output = json.loads(json_output)

In [35]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 19
Multiple chunk questions: 12
Impossible questions: 5


In [39]:
points = qdrant_client.scroll(collection_name="amazon-electronics-items-collection-01", 
 scroll_filter=Filter(must=[FieldCondition(key="parent_asin", match=MatchValue(value="B08DLDCJQR"))]))[0]

In [40]:
points

[Record(id=4, payload={'preprocessed_description': 'GE UltraPro 6 Outlet Surge Protector, 2 ft Designer Braided Extension Cord, Flat Plug, Long Power Cord, Wall Mount, Black, 44070 Power more - Expand the reach of your outlet to power all your devices in one convenient location, either on the floor or on the wall thanks to the wall mount Keyholes of the GE ultrapure 6 outlet surge protector Flat Plug - save space behind furniture and appliances and keep other outlets accessible with the slim design of the low-profile, angled, flat Plug Extension cord - The 2-foot braided extension cord on the surge protector power strip helps to keep cords tangle-free Protection - Electronics are protected with a 620 joules protection rating, integrated circuit breaker and automatic shutdown technology Trusted Brand - GE is the #1 Brand in surge protection', 'image': 'https://m.media-amazon.com/images/I/41Bt6VJ6omL._AC_.jpg', 'rating_number': 2301, 'price': 9.88, 'average_rating': 4.7, 'parent_asin': '

In [43]:
def get_product_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(collection_name="amazon-electronics-items-collection-01", 
     scroll_filter=Filter(must=[FieldCondition(key="parent_asin", match=MatchValue(value=parent_asin))]))[0]
    return points[0].payload['preprocessed_description']

In [44]:
get_product_description("B08DLDCJQR")

'GE UltraPro 6 Outlet Surge Protector, 2 ft Designer Braided Extension Cord, Flat Plug, Long Power Cord, Wall Mount, Black, 44070 Power more - Expand the reach of your outlet to power all your devices in one convenient location, either on the floor or on the wall thanks to the wall mount Keyholes of the GE ultrapure 6 outlet surge protector Flat Plug - save space behind furniture and appliances and keep other outlets accessible with the slim design of the low-profile, angled, flat Plug Extension cord - The 2-foot braided extension cord on the surge protector power strip helps to keep cords tangle-free Protection - Electronics are protected with a 620 joules protection rating, integrated circuit breaker and automatic shutdown technology Trusted Brand - GE is the #1 Brand in surge protection'

### create eval dataset in langsmith

In [45]:
from langsmith import Client
from dotenv import load_dotenv

load_dotenv('../../.env')
ls_client = Client()


In [46]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

In [47]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_product_description(id) for id in item["chunk_ids"]]
        }
    )